# Linear Regression

---

## Overview

In **regression**, we are given labeled data where the target values are real-valued. The goal is to learn a function $\hat{y} = f(\mathbf{x})$ that approximates the true target $y$.

**Linear regression** assumes the target is a linear function of the features:

$$\hat{y} = \mathbf{w} \cdot \mathbf{x} + b$$

where $\mathbf{w}$ is the weight vector and $b$ is the bias term.

---

## Cost Function

We minimize the **Mean Squared Error (MSE)**:

$$C(\mathbf{w}, b) = \frac{1}{2N} \sum_{i=1}^{N} \left(\hat{y}^{(i)} - y^{(i)}\right)^2$$

## Closed-Form Solution (Normal Equation)

$$\mathbf{w} = (X^\top X)^{-1} X^\top \mathbf{y}$$

---

**Dataset:** Gym Members Exercise Tracking (`gym_members_exercise_tracking.csv`)  
**Task:** Predict `Calories_Burned` from exercise features.
except FileNotFoundError:
    from sklearn.datasets import make_regression
    X, y = make_regression(n_samples=500, n_features=5, noise=10, random_state=42)
    print('CSV not found. using synthetic regression data')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme()

from rice_ml.supervised_learning import LinearRegression
from rice_ml.preprocess import StandardScaler, train_test_split
from rice_ml.metrics import mse, rmse, r2_score

In [ ]:
# Load dataset. falls back to synthetic data if CSV is not present
try:
    df = pd.read_csv('../../../data/gym_members_exercise_tracking.csv')
    features = ['Age', 'Weight (kg)', 'Session_Duration (hours)', 'Workout_Frequency (days/week)', 'Fat_Percentage']
    target = 'Calories_Burned'
    df = df[features + [target]].dropna()
    X = df[features].values.astype(float)
    y = df[target].values.astype(float)
    print(f'Loaded gym dataset: {X.shape[0]} samples, {X.shape[1]} features')
except FileNotFoundError:
    from sklearn.datasets import make_regression
    X, y = make_regression(n_samples=300, n_features=5, noise=50, random_state=0)
    print('CSV not found. using synthetic regression data')

print(f'X shape: {X.shape}, y range: [{y.min():.1f}, {y.max():.1f}]')

In [ ]:
# Preprocess
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} samples | Test: {len(X_test)} samples')

## Fit. Normal Equation

The `LinearRegression` class solves $\mathbf{w} = (X^\top X)^{-1} X^\top \mathbf{y}$ using `np.linalg.lstsq`.

In [ ]:
model = LinearRegression(method='normal')
model.fit(X_train, y_train)

print(f'Weights: {model.w_[:-1]}')
print(f'Bias:    {model.w_[-1]:.4f}')

In [ ]:
# Evaluate
y_pred = model.predict(X_test)

print(f'Test MSE:  {mse(y_test, y_pred):.2f}')
print(f'Test RMSE: {rmse(y_test, y_pred):.2f}')
print(f'Test R²:   {r2_score(y_test, y_pred):.4f}')

In [ ]:
# Actual vs Predicted
plt.figure(figsize=(10, 8))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', label='Perfect fit')
plt.xlabel('Actual', fontsize=15)
plt.ylabel('Predicted', fontsize=15)
plt.title('Linear Regression: Actual vs Predicted', fontsize=18)
plt.legend(fontsize=13)
plt.show()

In [ ]:
# Residuals plot: residual = y_true - y_pred
# A good model shows residuals randomly scattered around 0 with no pattern
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs predicted
axes[0].scatter(y_pred, residuals, alpha=0.6, color='steelblue', edgecolors='white', s=50)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Predicted', fontsize=13)
axes[0].set_ylabel('Residual (y - y_hat)', fontsize=13)
axes[0].set_title('Residuals vs Predicted', fontsize=14)

# Residual distribution
axes[1].hist(residuals, bins=25, color='salmon', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual', fontsize=13)
axes[1].set_ylabel('Count', fontsize=13)
axes[1].set_title('Residual Distribution', fontsize=14)

plt.suptitle('Residual Analysis: Normal Equation', fontsize=16)
plt.tight_layout()
plt.show()

print(f'Mean residual:   {residuals.mean():.4f}  (should be ~0)')
print(f'Std of residuals:{residuals.std():.4f}')


In [ ]:
# Compare: SGD vs Normal Equation
model_sgd = LinearRegression(method='sgd', alpha=0.01, epochs=500)
model_sgd.fit(X_train, y_train)

plt.figure(figsize=(10, 6))
plt.plot(model_sgd.errors_, color='steelblue')
plt.xlabel('Epoch', fontsize=15)
plt.ylabel('MSE Cost', fontsize=15)
plt.title('SGD Training Cost per Epoch', fontsize=18)
plt.show()

y_pred_sgd = model_sgd.predict(X_test)
print(f'SGD Test R²:    {r2_score(y_test, y_pred_sgd):.4f}')
print(f'Normal Test R²: {r2_score(y_test, y_pred):.4f}')

## Interpretation

- The **normal equation** gives the exact least-squares solution in one step.
- **SGD** approximates the same solution iteratively. useful for very large datasets.
- $R^2$ close to 1 indicates the model explains most of the variance in calories burned.
- Points along the red diagonal in the scatter plot indicate accurate predictions.